In [9]:
import matplotlib.pyplot as plt
import io
from PIL import Image, ImageChops

white = (255, 255, 255, 255)

def latex_to_img(tex):
    buf = io.BytesIO()
    plt.rc('text', usetex=True)
    plt.rc('font', family='serif')
    plt.axis('off')
    plt.text(0.05, 0.5, f'${tex}$', size=40)
    plt.savefig(buf, format='png')
    plt.close()

    im = Image.open(buf)
    bg = Image.new(im.mode, im.size, white)
    diff = ImageChops.difference(im, bg)
    diff = ImageChops.add(diff, diff, 2.0, -100)
    bbox = diff.getbbox()
    return im.crop(bbox)

In [10]:
from scipy.stats import wilcoxon
import pandas as pd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
pd.options.display.float_format = '{:.5f}'.format

def wilcoxon_test(results_best,results_compare):
    pvalue = wilcoxon(x=results_best,y=results_compare).pvalue
    if pvalue<=0.001:
        return {}
    elif pvalue>0.001 and pvalue<0.05:
        return {'dag':'--rwrap'}
    else:
        return {'ddag':'--rwrap'}

In [11]:
import pandas as pd
import os

export_dir = 'latex'
path = "results"

def show_results_table(experiment_names, base_path='', include_std=False, row_names=None, error_measures=['RAE']):
    table = pd.DataFrame(columns=error_measures, dtype='float')
    results_error = {em: {} for em in error_measures}

    available_experiments = []

    for experiment in experiment_names:
        path_errors = os.path.join(base_path, path, experiment + '_errors.txt')
        if os.path.exists(path_errors):
            try:
                results = pd.read_csv(path_errors)
                available_experiments.append(experiment)
                for error_measure in error_measures:
                    if error_measure in results:
                        results_error[error_measure][experiment] = results[error_measure]
                        table.loc[experiment, error_measure] = results[error_measure].mean()
            except Exception as e:
                print(f"Could not read {path_errors}: {e}")
        else:
            print(f"File {path_errors} does not exist.")

    if table.empty:
        return "No results available yet.", results_error

    best_method = {}
    for error_measure in error_measures:
        if table[error_measure].notna().any():
            best_method[error_measure] = table[error_measure].idxmin()

    table_style = table.style

    # Apply Wilcoxon test (if available)
    for experiment in available_experiments:
        for error_measure in error_measures:
            if experiment != best_method.get(error_measure) and experiment in results_error[error_measure]:
                try:
                    # Ensure both experiment and best method have results
                    base_results = results_error[error_measure].get(best_method[error_measure])
                    if base_results is not None:
                        table_style = table_style.set_properties(
                            subset=pd.IndexSlice[experiment, error_measure],
                            **wilcoxon_test(results_error[error_measure][experiment], base_results)
                        )
                except Exception as e:
                    print(f"Wilcoxon test failed for {experiment} vs {best_method[error_measure]}: {e}")

    for error_measure in error_measures:
        if error_measure in table.columns:
            table_style = table_style.highlight_min(axis=0, props='textbf:--rwrap;', subset=error_measure)

    def add_deviation(x, std):
        return "{:.5f}".format(x) + ' $\pm$ ' + "{:.5f}".format(std)

    if include_std:
        for experiment in available_experiments:
            for error_measure in error_measures:
                if experiment in results_error[error_measure]:
                    std_value = results_error[error_measure][experiment].std()
                    table_style = table_style.format(
                        formatter=lambda x, std=std_value: add_deviation(x, std),
                        subset=pd.IndexSlice[experiment, error_measure]
                    )

    latex_code = table_style.to_latex(hrules=True, column_format="r|" + "r" * len(table.columns))
    if row_names is not None:
        for experiment_name, row_name in zip(experiment_names, row_names):
            latex_code = latex_code.replace(experiment_name, row_name)

    return latex_code.replace("_", "\\_"), results_error


## Results CIFAR 10

In [12]:
experiment_names = ["cifar10_CC_mrae","cifar10_PCC_mrae","cifar10_ACC_mrae","cifar10_PACC_mrae","cifar10_DMy_mrae","cifar10_EMQ_mrae","cifar10_EMQ-Platt_mrae","cifar10_deepsets_median_final","cifar10_deepsets_max_final","cifar10_deepsets_avg_final","cifar10_histnet_final","cifar10_gmnet_final","cifar10_gmnet_01_final","cifar10_gmnet_001_final","cifar10_gmnet_0001_final"]
row_names = ["CC","PCC","ACC","PACC","DMy","EMQ","EMQ-Platt","DeepSets (Median)","DeepSets (Max)","DeepSets (Avg)","HistNetQ","GMNet","GMNet (reg 0.1)","GMNet (reg 0.01)","GMNet (reg 0.001)"]
table,_ = show_results_table(experiment_names=experiment_names, base_path='', include_std=True,error_measures=['RAE'], row_names=row_names)
print(table)
with open(os.path.join(export_dir,'tables/cifar10.tex'),'w') as f:
    f.write(table)

\begin{tabular}{r|r}
\toprule
 & RAE \\
\midrule
CC & 0.52963 $\pm$ 0.69583 \\
PCC & 0.60844 $\pm$ 0.74459 \\
ACC & 0.36017 $\pm$ 0.47573 \\
PACC & 0.35578 $\pm$ 0.41790 \\
DMy & 0.21438 $\pm$ 0.23418 \\
EMQ & \ddag{0.18864 $\pm$ 0.20515} \\
EMQ-Platt & 0.50291 $\pm$ 0.59707 \\
DeepSets (Median) & 0.22072 $\pm$ 0.23906 \\
DeepSets (Max) & 0.93907 $\pm$ 0.64273 \\
DeepSets (Avg) & 0.22114 $\pm$ 0.22161 \\
HistNetQ & 0.22712 $\pm$ 0.24774 \\
GMNet & \ddag{0.17275 $\pm$ 0.14382} \\
GMNet (reg 0.1) & 0.18189 $\pm$ 0.14567 \\
GMNet (reg 0.01) & \textbf{0.16840 $\pm$ 0.12279} \\
GMNet (reg 0.001) & \ddag{0.17233 $\pm$ 0.14956} \\
\bottomrule
\end{tabular}

